## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 46.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 316.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 329.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 189.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 310.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.4 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.3.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# For installing the libraries & downloading models from HF Hub
!pip install huggingface_hub==0.23.2 pandas==1.5.3 tiktoken==0.6.0 pymupdf==1.25.1 langchain==0.1.1 langchain-community==0.0.13 chromadb==0.4.22 sentence-transformers==2.3.1 numpy==1.25.2 -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
#from langchain.text_splitter import RecursiveCharacterTextSplitter
#from langchain_community.document_loaders import PyMuPDFLoader
#from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
#from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

## Question Answering using LLM

#### Downloading and Loading the model

In [ ]:
model_name_or_path = "TheBloke/Llama-2-13B-chat-GGUF"
model_basename = "llama-2-13b-chat.Q5_K_M.gguf" # the model is in gguf format

In [ ]:
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

In [ ]:
llm = Llama(
    model_path=model_path,
    n_ctx=1024,
)

AVX = 1 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


#### Response

In [ ]:
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
response(user_input)

'\nSepsis is a life-threatening condition that can arise from an infection, and it is a leading cause of death in hospitals. The management of sepsis in a critical care unit (CCU) requires a systematic approach that includes early recognition, prompt treatment, and ongoing monitoring. Here is a general protocol for managing sepsis in a CCU:\n\n1. Early recognition:\n\t* Use a standardized screening tool, such as the Systemic Inflammatory Response Syndrome (SIRS) criteria or the Sepsis-3 definition'

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(user_input)

Llama.generate: prefix-match hit


'\n\nAnswer:\n\nAppendicitis is a medical emergency that requires prompt treatment. The most common symptoms of appendicitis include:\n\n1. Severe pain in the abdomen, usually starting near the belly button and then moving to the lower right side of the abdomen.\n2. Nausea and vomiting.\n3. Loss of appetite.\n4. Fever.\n5. Abdominal tenderness and guarding (muscle tension).\n6. Abdominal swelling.\n7. Diarrhea or constipation'

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_input)

Llama.generate: prefix-match hit


'\nSudden patchy hair loss, also known as alopecia areata, can be caused by a variety of factors. Here are some effective treatments and solutions for addressing this condition:\n\n1. Corticosteroid injections: These injections can help suppress the immune system and promote hair growth. They are usually given every 4-6 weeks and can be effective in promoting hair regrowth.\n2. Topical corticosteroids: Over-the-counter or prescription topical corticosteroids can be applied directly to the affected area to reduce inflamm'

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_input)

Llama.generate: prefix-match hit


"\nThere are several treatments that may be recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function. The specific treatment plan will depend on the location and severity of the injury, as well as the individual's overall health and medical history. Some common treatments for brain injuries include:\n\n1. Medications: To manage symptoms such as pain, inflammation, and anxiety.\n2. Rehabilitation therapy: To help regain lost function and improve cognitive, physical, and emotional abilities."

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(user_input)

Llama.generate: prefix-match hit


'\n\nA person who has fractured their leg during a hiking trip requires prompt medical attention to ensure proper healing and prevent any further complications. Here are some necessary precautions and treatment steps:\n\n1. Stop all activity: The first step is to stop all activity and rest the affected leg immediately. This will help prevent further damage and reduce the risk of complications.\n2. Seek medical attention: It is essential to seek medical attention as soon as possible, especially if the fracture is displaced or if there are any signs of nerve or blood vessel damage. A healthcare'

Observations:



*   LLM model is taking more time to retrive the answer for the given questions
*   LLM model response are too verbose and not specific enough



## Question Answering using LLM with Prompt Engineering

In [ ]:
system_message = """
You are an experienced medical assistant designed to provide accurate, evidence-based clinical information.
Follow these rules:

1. Use only medically verified knowledge.
2. Provide concise, step-by-step answers.
3. Use clinical terminology used in standard medical guidelines.
4. If you don’t know the answer, say 'Information not available.'
"""

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = system_message + "\n" + "What is the protocol for managing sepsis in a critical care unit?"
response(user_input, 200, 0, 0.95, 50)

Llama.generate: prefix-match hit


'\n\nAnswer:\n\nSepsis management in a critical care unit should be guided by the Surviving Sepsis Campaign (SSC) guidelines. The SSC recommends the following protocol:\n\n1. Early recognition and diagnosis of sepsis: Assess patients for signs and symptoms of sepsis, including fever, tachycardia, tachypnea, and organ dysfunction.\n2. Rapid administration of antibiotics: Start broad-spectrum antibiotics within the first hour of recognition of sepsis, and administer them promptly to all patients with suspected severe infection.\n3. Vasopressor support: Use vasopressors to maintain mean arterial pressure (MAP) ≥65 mmHg and serum lactate level <2 mmol/L.\n4. Oxygen therapy: Provide oxygen ther'

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input = system_message + "\n" + "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(user_input, 180, 0.2, 0.9, 40)

Llama.generate: prefix-match hit


''

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input = system_message + "\n" + "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_input, 180, 0, 0.8, 40)

'\n\nAnswer:\n\nSudden patchy hair loss, also known as alopecia areata, can be caused by various factors. Here are some effective treatments and solutions for addressing this condition:\n\n1. Corticosteroid injections or topical creams: These medications can help suppress the immune system and promote hair growth. They are usually prescribed for mild to moderate cases of alopecia areata.\n2. Minoxidil (Rogaine): This over-the-counter solution is applied directly to the affected area and can help stimulate hair growth and slow down hair loss.\n3. Anthralin (Psoralen): This medication is used to treat skin conditions and can also promote hair growth in some cases of alopecia areata.\n4. Phototherapy: Ex'

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input = system_message + "\n" + "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_input, 190, 0.1, 0.95, 30)

Llama.generate: prefix-match hit


'\n\nPlease provide step-by-step recommendations based on standard medical guidelines.'

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input = system_message + "\n" + "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(user_input, 170, 0, 0.95, 50)

Llama.generate: prefix-match hit


'\n\nAnswer:\n\nPrecautions:\n\n1. Immobilize the affected limb to prevent further injury or displacement of the fracture.\n2. Apply ice packs to reduce swelling and pain.\n3. Elevate the affected limb above heart level to reduce bleeding and swelling.\n4. Monitor vital signs, especially pulse and breathing rate.\n5. Keep the patient warm and dry to prevent hypothermia.\n6. Provide pain management as needed.\n7. Stabilize the fracture site with a splint or cast.\n8. Transport the patient to a medical facility as soon as possible.\n\nTreatment Steps:\n\n1. Perform a thorough physical examination and assess the extent of the'

Observations:



*   LLM with Prompt engineering still taking more time to retrieve the response
*   But the answers are more precise than using the plain LLM



## Data Preparation for RAG

### Loading the Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
merck_pdf_path = "/content/drive/MyDrive/Colab Notebooks/content/medical_diagnosis_manual.pdf"

In [ ]:
#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter



In [ ]:
!pip install langchain langchain-community
!pip install pymupdf

INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.3.4 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 98.1 MB/s eta 0:00:00


In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

In [ ]:
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

In [ ]:
pdf_loader = PyMuPDFLoader(merck_pdf_path)


In [ ]:
medical_manual = pdf_loader.load()

### Data Overview

#### Checking the first 5 pages

In [ ]:
for i in range(6):
    print(f"Page Number : {i+1}",end="\n")
    print(medical_manual[i].page_content,end="\n")

Page Number : 1
sindhu13.c@gmail.com
DP2WE13I8M
eant for personal use by sindhu13.c@gm
shing the contents in part or full is liable
Page Number : 2
sindhu13.c@gmail.com
DP2WE13I8M
This file is meant for personal use by sindhu13.c@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ...........................................................................................................................................................................................
53
1

#### Checking the number of pages

In [ ]:
len(medical_manual)

4114

### Data Chunking

In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=600,
    chunk_overlap= 80
)

In [ ]:
document_chunks = pdf_loader.load_and_split(text_splitter)


In [ ]:
len(document_chunks)


7904

In [ ]:
document_chunks[0].page_content


'sindhu13.c@gmail.com\nDP2WE13I8M\neant for personal use by sindhu13.c@gm\nshing the contents in part or full is liable'

### Embedding

In [ ]:
embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')


In [ ]:
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [ ]:
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  1024


True

### Vector Database

In [ ]:
out_dir = 'medical_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [ ]:
!pip install chromadb


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 121.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 107.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 121.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 12.6 MB/

In [ ]:
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=out_dir
)

In [ ]:
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)


In [ ]:
vectorstore.embeddings


HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='thenlper/gte-large', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [ ]:
vectorstore.similarity_search("Etiology",k=3)

[Document(metadata={'page': 2841, 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'total_pages': 4114, 'modDate': 'D:20251103064740Z', 'file_path': '/content/drive/MyDrive/Colab Notebooks/content/medical_diagnosis_manual.pdf', 'trapped': '', 'creationDate': 'D:20120615054440Z', 'moddate': '2025-11-03T06:47:40+00:00', 'creator': 'Atop CHM to PDF Converter', 'creationdate': '2012-06-15T05:44:40+00:00', 'format': 'PDF 1.7', 'subject': '', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'source': '/content/drive/MyDrive/Colab Notebooks/content/medical_diagnosis_manual.pdf', 'author': '', 'keywords': ''}, page_content='Etiology\nEtiology is unknown; however, risk factors include the following:\n• Nulliparity\n• Preexisting chronic hypertension\n• Vascular disorders (eg, renal disorders, diabetic vasculopathy)\n• Pregestational or gestational diabetes\n• Older (> 35) or very young (eg, < 17) maternal age\nThe Merck Manual of Diagnosis & Therapy, 19th Edition\nCh

### Retriever

In [ ]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 2}
)

In [ ]:
rel_docs = retriever.get_relevant_documents("what is Renal Atheroembolism?")
rel_docs

[Document(metadata={'page': 2593, 'total_pages': 4114, 'moddate': '2025-11-03T06:47:40+00:00', 'trapped': '', 'author': '', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'source': '/content/drive/MyDrive/Colab Notebooks/content/medical_diagnosis_manual.pdf', 'creationDate': 'D:20120615054440Z', 'format': 'PDF 1.7', 'modDate': 'D:20251103064740Z', 'creator': 'Atop CHM to PDF Converter', 'keywords': '', 'subject': '', 'creationdate': '2012-06-15T05:44:40+00:00', 'file_path': '/content/drive/MyDrive/Colab Notebooks/content/medical_diagnosis_manual.pdf', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition'}, page_content='but not in bilateral renal artery stenosis. These drugs can reduce GFR and increase serum BUN and\ncreatinine levels. If GFR decreases enough to increase serum creatinine, Ca channel blockers (eg,\namlodipine, felodipine) or vasodilators (eg, hydralazine, minoxidil) should be added or substituted (see p.\n2071).\nRenal Atheroembolism\nRenal ather

In [ ]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [ ]:
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

mistral-7b-instruct-v0.2.Q6_K.gguf:   0%|          | 0.00/5.94G [00:00<?, ?B/s]

In [ ]:
llm = Llama(
    model_path=model_path,
    n_ctx=5000,
    n_gpu_layers=38,
    n_batch=512
)

AVX = 1 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


In [ ]:
llm("what is Renal Atheroembolism?")['choices'][0]['text']

'\n\nRenal atheroembolism (RAE) is a condition where small emboli (clots or particles) originating from the aorta travel to the kidneys, causing damage and sometimes impairing their function. The emboli typically contain cholesterol, calcium, and other debris that can cause inflammation and necrosis in the renal artery or arterioles.\n\nRAE is most commonly seen in patients with atherosclerosis, which is a buildup of plaque (fatty deposits) in the ar'

### System and User Prompt Template

In [ ]:
qna_system_message = """
You are an assistant whose work is to review the report and provide the appropriate answers from the context.
User input will have the context required by you to answer user questions.
This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: ###Question.

Please answer only using the context provided in the input. Do not mention anything about the context in your final answer.

If the answer is not found in the context, respond "I don't know".
"""

In [ ]:
qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}

###Question
{question}
"""

### Response Function

In [ ]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
print(generate_rag_response(user_input))

Llama.generate: prefix-match hit


Based on the context provided, the protocol for managing sepsis in a critical care unit includes:
1. Prompt empiric antibiotic therapy: Start immediately after suspecting sepsis.
2. Antibiotic selection: Based on suspected source, clinical setting, knowledge or suspicion of causative organisms and sensitivity patterns common to that specific inpatient unit, and previous culture results.
3. Regimen for septic shock of unknown cause: Gentamicin or tobramycin 5.1 mg/kg IV once/day plus a 3rd-generation cephalospor


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(generate_rag_response(user_input))

Llama.generate: prefix-match hit


###Answer
The common symptoms for appendicitis include abdominal pain, anorexia, and abdominal tenderness. Appendicitis cannot be cured via medicine alone; surgery, specifically appendectomy, is required for treatment.


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(generate_rag_response(user_input))

Llama.generate: prefix-match hit


Based on the context provided, the possible causes of sudden patchy hair loss include alopecia areata. The effective treatments or solutions for addressing this condition include topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). It is important to note that the choice of treatment depends on the severity of the condition. Additionally, hormonal modulators such as oral


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(generate_rag_response(user_input))

Llama.generate: prefix-match hit


Based on the context provided, the recommended treatments for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function include:
1. Ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure.
2. Surgery to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas.
3. Maintaining adequate brain perfusion and oxygenation in the first few days after the injury and preventing complications of altered


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(generate_rag_response(user_input))

Llama.generate: prefix-match hit


The context mentions that for a fracture, symptoms include pain, swelling, ecchymosis, crepitation, deformity, and abnormal motion. The treatment involves analgesics, immobilization, and sometimes surgery. Immobilization is important to prevent further injury and facilitate healing. A cast or splint may be used depending on the severity of the injury and the length of immobilization required. The injured limb should be elevated above the heart for the first 2 days to minimize swelling. After 48 hours, periodic application of warmth may relieve pain and speed healing


### Fine-tuning

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
print(generate_rag_response(user_input),4,256,0.0,1.0,20)

Llama.generate: prefix-match hit


Based on the context provided, the protocol for managing sepsis in a critical care unit includes:
1. Prompt empiric antibiotic therapy: Start immediately after suspecting sepsis.
2. Antibiotic selection: Based on suspected source, clinical setting, knowledge or suspicion of causative organisms and sensitivity patterns common to that specific inpatient unit, and previous culture results.
3. Regimen for septic shock of unknown cause: Gentamicin or tobramycin 5.1 mg/kg IV once/day plus a 3rd-generation cephalospor 4 256 0.0 1.0 20


In [ ]:
user_input = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(generate_rag_response(user_input),4,256,0.2,0.95,40)

Llama.generate: prefix-match hit


###Answer
The common symptoms for appendicitis include abdominal pain, anorexia, and abdominal tenderness. Appendicitis cannot be cured via medicine alone; surgery, specifically appendectomy, is required for treatment. 4 256 0.2 0.95 40


In [ ]:
user_input = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(generate_rag_response(user_input),6,400,0.1,0.90,50)

Llama.generate: prefix-match hit


Based on the context provided, the possible causes of sudden patchy hair loss include alopecia areata. The effective treatments or solutions for addressing this condition include topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). It is important to note that the choice of treatment depends on the severity of the condition. Additionally, hormonal modulators such as oral 6 400 0.1 0.9 50


In [ ]:
user_input = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(generate_rag_response(user_input),2,200,0.0,0.8,20)

Llama.generate: prefix-match hit


Based on the context provided, the recommended treatments for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function include:
1. Ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure.
2. Surgery to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas.
3. Maintaining adequate brain perfusion and oxygenation in the first few days after the injury and preventing complications of altered 2 200 0.0 0.8 20


In [ ]:
user_input = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(generate_rag_response(user_input),5,350,0.7,0.95,100)

Llama.generate: prefix-match hit


The context mentions that for a fracture, symptoms include pain, swelling, ecchymosis, crepitation, deformity, and abnormal motion. The treatment involves analgesics, immobilization, and sometimes surgery. Immobilization is important to prevent further injury and facilitate healing. A cast or splint may be used depending on the severity of the injury and the length of immobilization required. The injured limb should be elevated above the heart for the first 2 days to minimize swelling. After 48 hours, periodic application of warmth may relieve pain and speed healing 5 350 0.7 0.95 100


Observations:



*   RAG Model performance is best when compared to other 2 models (LLM , LLM with prompt engineering)
*   RAG Model response is more precise from the context provided



## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [ ]:
groundedness_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context

Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluaton criteria and assign a score.
"""

In [ ]:
relevance_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context.
Consider whether all and only the important aspects are contained in the answer when evaluating relevance.

Instructions:
1. First write down the steps that are needed to evaluate the context as per the metric.
2. Give a step-by-step explanation if the context adheres to the metric considering the question as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the context using the evaluaton criteria and assign a score.
"""

In [ ]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [ ]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
ground,rel = generate_ground_relevance_response(user_input,max_tokens=350)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the key information related to managing sepsis in a critical care unit from the context.
2. Compare the AI generated answer with the identified key information.
3. Check if the answer is derived only from the information presented in the context.

Explanation:
The AI generated answer includes giving parenteral antibiotics after taking specimens for Gram stain and culture, starting very prompt empiric therapy as soon as sepsis is suspected, changing the antibiotic regimen based on culture and sensitivity results, continuing antibiotics for at least 5 days after shock resolves and evidence of infection subsides, draining abscesses or excising necrotic tissues, and normalizing blood glucose to improve outcome in critically ill patients.

The context provides information on the importance of monitoring and testing in critical care units, including blood tests and cardiac monitoring. It also mentions that ICU patients typically have routine daily b

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
ground,rel = generate_ground_relevance_response(user_input,max_tokens=350)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the information in the context related to appendicitis and its symptoms.
2. Determine if the AI generated answer includes only the information derived from the context.
3. Check if the AI generated answer mentions that appendicitis cannot be cured via medicine alone and that surgical removal is required for treatment.

Explanation:
The context provides information about the symptoms of appendicitis, which include abdominal pain, anorexia, and abdominal tenderness. The AI generated answer correctly identifies these symptoms as common for appendicitis. Additionally, the answer states that appendicitis cannot be cured via medicine alone and that surgical removal is required for treatment. This information is also present in the context.

Therefore, the answer adheres to the metric as it is derived only from the information presented in the context.

Rating:
Based on the evaluation criteria, I would rate this answer as a 5 because the metric is fo

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
ground,rel = generate_ground_relevance_response(user_input,max_tokens=350)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the main question and the specific information being asked for in the question.
2. Read through the context provided to understand the key points related to the topic of sudden patchy hair loss, including potential causes and treatments.
3. Determine if the AI generated answer is derived solely from the information presented in the context.

Explanation:
The AI generated answer adheres to the metric as it mentions only the treatments and causes discussed in the context. The answer does not introduce any new or irrelevant information.

Evaluation:
Based on the evaluation criteria, I would rate the answer a 5 since it follows the metric completely by deriving all the information from the context provided.

 Steps to evaluate the context as per the metric:
1. Identify the main aspects of the question: effective treatments or solutions for sudden patchy hair loss and possible causes.
2. Determine if the context provides information on both aspects

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
ground,rel = generate_ground_relevance_response(user_input,max_tokens=350)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the key information related to treatments for a person with a traumatic brain injury (TBI) from the context.
2. Compare the identified information with the AI generated answer to check if the answer is derived only from the context.

Explanation:
The AI generated answer mentions that there is no specific treatment mentioned in the context, but supportive care should include preventing systemic complications due to immobilization, providing good nutrition, and preventing pressure ulcers. This information is directly taken from the context. Therefore, the metric is followed completely.

Rating:
Based on the evaluation criteria, I would rate the answer as 5 (The metric is followed completely).

 Steps to evaluate the context as per the relevance metric:
1. Identify the main aspects of the question: The question asks about treatments for a person with a brain injury that temporarily or permanently impairs brain function.
2. Determine if all import

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
ground,rel = generate_ground_relevance_response(user_input,max_tokens=350)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the information in the context related to precautions and treatment steps for a person with a fractured leg.
2. Compare the information in the context to the answer to ensure that the answer is derived only from the context.
3. Evaluate the extent to which the answer follows the metric by determining if all the necessary precautions and treatment steps are included and if they are directly taken from the context.

Explanation:
The answer includes several precautions and treatment steps for a person with a fractured leg, such as keeping the injured limb elevated above heart level, applying periodic warmth using a heating pad, immobilizing joints proximal and distal to the injury, avoiding putting objects inside the cast or getting it wet, inspecting the edges and skin around the cast daily, applying lotion to any red or sore areas, padding rough cast edges, seeking medical care if an odor emanates from within the cast or a fever develops, maint

## Actionable Insights and Business Recommendations




*   LLM Only Model

1. It generates genric ansewers without any reference document
2. This is not suitable for medical industry because of high risk in medical environment


*   LLM with Prompt Engineering

1. It summarize the answers better
2. But still not acceptable in medical industry becuase of safety compliance

*   RAG

1. This model answers are stronglt tied with retrieved text from PDF
2. This is best for answering questions in medical industry








<font size=6 color='blue'>Power Ahead</font>
___